# Imports

This notebook establishes the foundation for the Rhabdomyosarcoma TMB & Survival Explorer by loading, validating, and cleaning the clinical dataset. The objective is to ensure that all subsequent exploratory analyses, visualizations, and survival analyses are performed using a consistent and well-documented dataset.

# Objectives: 
* Load the clinical dataset from the project data/ directory.
* Verify that all required variables are present for downstream analyses.
* Inspect the structure, data types, and completeness of the dataset.
* Quantify missing values and identify potential data quality issues.
* Standardize survival status by creating a binary event indicator for survival analysis.
* Generate cleaned sample-level datasets for exploratory analyses.
* Prepare a survival-ready dataset by removing observations with incomplete survival information.
* Evaluate the number of samples per patient and identify patients with multiple tumor samples.
* Assess consistency of survival information across multiple samples from the same patient.
* Quantify within-patient variation in tumor mutational burden (TMB) to guide future patient-level aggregation methods.
* Save cleaned datasets for reuse in later notebooks and the final Streamlit application.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# Define the project and data paths 

In [2]:
PROJECT_DIR = Path.cwd().parent
DATA_FILE = PROJECT_DIR / "data" / "soft_tissue_msk_2023_clinical_data.tsv"

print("Project directory:", PROJECT_DIR)
print("Data file:", DATA_FILE)
print("File exists:", DATA_FILE.exists())

Project directory: /users/PAS3421/emmafischels/Capstone
Data file: /users/PAS3421/emmafischels/Capstone/data/soft_tissue_msk_2023_clinical_data.tsv
File exists: True


# Load the dataset

In [3]:
df = pd.read_csv(DATA_FILE, sep="\t")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 42
Columns: 27


,Study ID,Patient ID,Sample ID,Age at Which Sequencing was Reported (Years),Cancer Type,Cancer Type Detailed,Ethnicity Category,Fraction Genome Altered,Gene Panel,Institute Source,...,Primary Tumor Site,Race Category,Sample Class,Number of Samples Per Patient,Sample coverage,Sample Type,Sex,Somatic Status,TMB (nonsynonymous),Tumor Purity
0,soft_tissue_msk_2023,P-0006113,P-0006113-T01-IM5,18.0,Soft Tissue Sarcoma,Alveolar Rhabdomyosarcoma,"Spanish NOS; Hispanic NOS, Latino NOS",0.0844,IMPACT410,MSKCC,...,Soft Tissue,WHITE,Tumor,2,949,Primary,Male,Matched,0.978720,80.0
1,soft_tissue_msk_2023,P-0006113,P-0006113-T02-IM6,18.0,Soft Tissue Sarcoma,Alveolar Rhabdomyosarcoma,"Spanish NOS; Hispanic NOS, Latino NOS",0.3861,IMPACT468,MSKCC,...,Soft Tissue,WHITE,Tumor,2,743,Metastasis,Male,Matched,0.864698,30.0
2,soft_tissue_msk_2023,P-0013120,P-0013120-T04-IM6,19.0,Soft Tissue Sarcoma,Embryonal Rhabdomyosarcoma,Non-Spanish; Non-Hispanic,0.1217,IMPACT468,MSKCC,...,Soft Tissue,WHITE,Tumor,2,727,Metastasis,Female,Matched,0.000000,30.0
3,soft_tissue_msk_2023,P-0013120,P-0013120-T06-IM6,20.0,Soft Tissue Sarcoma,Embryonal Rhabdomyosarcoma,Non-Spanish; Non-Hispanic,0.2839,IMPACT468,NaN,...,Unknown,WHITE,Tumor,2,666,Metastasis,Female,Matched,0.000000,70.0
4,soft_tissue_msk_2023,P-0018770,P-0018770-T02-IM6,18.0,Soft Tissue Sarcoma,Embryonal Rhabdomyosarcoma,Non-Spanish; Non-Hispanic,0.6574,IMPACT468,MSKCC,...,Mediastinum,ASIAN-FAR EAST/INDIAN SUBCONT,Tumor,2,651,Primary,Male,Matched,0.864698,70.0


# Inspect the columns

In [4]:
df.columns.tolist()

['Study ID',
 'Patient ID',
 'Sample ID',
 'Age at Which Sequencing was Reported (Years)',
 'Cancer Type',
 'Cancer Type Detailed',
 'Ethnicity Category',
 'Fraction Genome Altered',
 'Gene Panel',
 'Institute Source',
 'Metastatic Site',
 'MSI Score',
 'MSI Type',
 'Mutation Count',
 'Oncotree Code',
 'Overall Survival (Months)',
 'Overall Survival Status',
 'Primary Tumor Site',
 'Race Category',
 'Sample Class',
 'Number of Samples Per Patient',
 'Sample coverage',
 'Sample Type',
 'Sex',
 'Somatic Status',
 'TMB (nonsynonymous)',
 'Tumor Purity']

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 27 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Study ID                                      42 non-null     object 
 1   Patient ID                                    42 non-null     object 
 2   Sample ID                                     42 non-null     object 
 3   Age at Which Sequencing was Reported (Years)  39 non-null     float64
 4   Cancer Type                                   42 non-null     object 
 5   Cancer Type Detailed                          42 non-null     object 
 6   Ethnicity Category                            40 non-null     object 
 7   Fraction Genome Altered                       42 non-null     float64
 8   Gene Panel                                    42 non-null     object 
 9   Institute Source                              39 non-null     objec

# Define required columns

In [6]:
REQUIRED_COLUMNS = [
    "Patient ID",
    "Sample ID",
    "TMB (nonsynonymous)",
    "Overall Survival (Months)",
    "Overall Survival Status",
]

missing_required_columns = [
    column for column in REQUIRED_COLUMNS
    if column not in df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Missing required columns: {missing_required_columns}"
    )

print("All required columns are present.")

All required columns are present.


# Examine missing values 

In [7]:
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("Missing Values")
    .to_frame()
)

missing_summary["Percent Missing"] = (
    missing_summary["Missing Values"] / len(df) * 100
).round(1)

missing_summary

,Missing Values,Percent Missing
Metastatic Site,27,64.3
Mutation Count,12,28.6
Tumor Purity,4,9.5
Age at Which Sequencing was Reported (Years),3,7.1
Institute Source,3,7.1
Ethnicity Category,2,4.8
Sex,2,4.8
MSI Score,2,4.8
MSI Type,2,4.8
Overall Survival Status,2,4.8


# Inspect survival status values

In [8]:
df["Overall Survival Status"].value_counts(dropna=False)

Overall Survival Status
1:DECEASED    22
0:LIVING      18
NaN            2
Name: count, dtype: int64

# Create a standardized event variable 

In [9]:
status_map = {
    "1:DECEASED": 1,
    "0:LIVING": 0,
}

df["Event"] = df["Overall Survival Status"].map(status_map)

df[
    [
        "Overall Survival Status",
        "Event",
    ]
].drop_duplicates()

,Overall Survival Status,Event
0,1:DECEASED,1.0
6,0:LIVING,0.0
10,NaN,NaN


# Check numeric survival and TMB columns

In [10]:
numeric_check_columns = [
    "TMB (nonsynonymous)",
    "Overall Survival (Months)",
]

df[numeric_check_columns].describe()

,TMB (nonsynonymous),Overall Survival (Months)
count,42.000000,40.000000
mean,1.528860,27.450400
std,1.460685,16.647167
min,0.000000,3.649000
25%,0.000000,14.984000
50%,1.309707,31.890000
75%,2.461042,37.570000
max,5.742431,69.008000


In [12]:
# Check for impossible values

for column in numeric_check_columns:
    print(
        column,
        "negative values:",
        (df[column] < 0).sum()
    )

TMB (nonsynonymous) negative values: 0
Overall Survival (Months) negative values: 0


# Create a clean sample-level dataset

In [13]:
sample_df = df.copy()

sample_df = sample_df.dropna(
    subset=[
        "Patient ID",
        "Sample ID",
        "TMB (nonsynonymous)",
    ]
)

print("Original rows:", len(df))
print("Clean sample-level rows:", len(sample_df))

sample_df.head()

Original rows: 42
Clean sample-level rows: 42


,Study ID,Patient ID,Sample ID,Age at Which Sequencing was Reported (Years),Cancer Type,Cancer Type Detailed,Ethnicity Category,Fraction Genome Altered,Gene Panel,Institute Source,...,Race Category,Sample Class,Number of Samples Per Patient,Sample coverage,Sample Type,Sex,Somatic Status,TMB (nonsynonymous),Tumor Purity,Event
0,soft_tissue_msk_2023,P-0006113,P-0006113-T01-IM5,18.0,Soft Tissue Sarcoma,Alveolar Rhabdomyosarcoma,"Spanish NOS; Hispanic NOS, Latino NOS",0.0844,IMPACT410,MSKCC,...,WHITE,Tumor,2,949,Primary,Male,Matched,0.978720,80.0,1.0
1,soft_tissue_msk_2023,P-0006113,P-0006113-T02-IM6,18.0,Soft Tissue Sarcoma,Alveolar Rhabdomyosarcoma,"Spanish NOS; Hispanic NOS, Latino NOS",0.3861,IMPACT468,MSKCC,...,WHITE,Tumor,2,743,Metastasis,Male,Matched,0.864698,30.0,1.0
2,soft_tissue_msk_2023,P-0013120,P-0013120-T04-IM6,19.0,Soft Tissue Sarcoma,Embryonal Rhabdomyosarcoma,Non-Spanish; Non-Hispanic,0.1217,IMPACT468,MSKCC,...,WHITE,Tumor,2,727,Metastasis,Female,Matched,0.000000,30.0,1.0
3,soft_tissue_msk_2023,P-0013120,P-0013120-T06-IM6,20.0,Soft Tissue Sarcoma,Embryonal Rhabdomyosarcoma,Non-Spanish; Non-Hispanic,0.2839,IMPACT468,NaN,...,WHITE,Tumor,2,666,Metastasis,Female,Matched,0.000000,70.0,1.0
4,soft_tissue_msk_2023,P-0018770,P-0018770-T02-IM6,18.0,Soft Tissue Sarcoma,Embryonal Rhabdomyosarcoma,Non-Spanish; Non-Hispanic,0.6574,IMPACT468,MSKCC,...,ASIAN-FAR EAST/INDIAN SUBCONT,Tumor,2,651,Primary,Male,Matched,0.864698,70.0,1.0


# Create the survival-ready dataset

In [14]:
sample_survival_df = df.dropna(
    subset=[
        "TMB (nonsynonymous)",
        "Overall Survival (Months)",
        "Event",
    ]
).copy()

sample_survival_df[
    [
        "Patient ID",
        "Sample ID",
        "TMB (nonsynonymous)",
        "Overall Survival (Months)",
        "Event",
    ]
].head()

,Patient ID,Sample ID,TMB (nonsynonymous),Overall Survival (Months),Event
0,P-0006113,P-0006113-T01-IM5,0.978720,37.085,1.0
1,P-0006113,P-0006113-T02-IM6,0.864698,37.085,1.0
2,P-0013120,P-0013120-T04-IM6,0.000000,31.890,1.0
3,P-0013120,P-0013120-T06-IM6,0.000000,31.890,1.0
4,P-0018770,P-0018770-T02-IM6,0.864698,32.449,1.0


# Check repeated patients 

In [15]:
samples_per_patient = (
    df.groupby("Patient ID")
    .size()
    .sort_values(ascending=False)
    .rename("Sample Count")
)

samples_per_patient

Patient ID
P-0060318    3
P-0035844    3
P-0006113    2
P-0013120    2
P-0019531    2
P-0018770    2
P-0028971    2
P-0022317    2
P-0045478    2
P-0048073    2
P-0050430    2
P-0033453    2
P-0051132    2
P-0051453    2
P-0063459    2
P-0068164    2
P-0069477    2
P-0073133    2
P-0074869    2
P-0077663    2
Name: Sample Count, dtype: int64

In [16]:
print("Unique patients:", df["Patient ID"].nunique())
print("Total samples:", len(df))
print("Patients with multiple samples:", (samples_per_patient > 1).sum())

Unique patients: 20
Total samples: 42
Patients with multiple samples: 20


# Check whether survival information is consistent within patients

In [17]:
survival_consistency = (
    df.groupby("Patient ID")
    .agg(
        survival_month_values=(
            "Overall Survival (Months)",
            lambda x: x.dropna().nunique()
        ),
        survival_status_values=(
            "Overall Survival Status",
            lambda x: x.dropna().nunique()
        ),
    )
)

inconsistent_survival = survival_consistency[
    (survival_consistency["survival_month_values"] > 1)
    | (survival_consistency["survival_status_values"] > 1)
]

inconsistent_survival

,survival_month_values,survival_status_values
Patient ID,,


# Check within-patient TMB variation

In [18]:
patient_tmb_summary = (
    df.groupby("Patient ID")
    .agg(
        sample_count=("Sample ID", "count"),
        tmb_mean=("TMB (nonsynonymous)", "mean"),
        tmb_median=("TMB (nonsynonymous)", "median"),
        tmb_min=("TMB (nonsynonymous)", "min"),
        tmb_max=("TMB (nonsynonymous)", "max"),
        tmb_std=("TMB (nonsynonymous)", "std"),
    )
    .reset_index()
)

patient_tmb_summary["tmb_range"] = (
    patient_tmb_summary["tmb_max"]
    - patient_tmb_summary["tmb_min"]
)

patient_tmb_summary.sort_values(
    "tmb_range",
    ascending=False
).head(10)

,Patient ID,sample_count,tmb_mean,tmb_median,tmb_min,tmb_max,tmb_std,tmb_range
11,P-0051132,2,2.461042,2.461042,0.000000,4.922083,3.480439,4.922083
16,P-0069477,2,1.230521,1.230521,0.000000,2.461042,1.740219,2.461042
2,P-0018770,2,1.729396,1.729396,0.864698,2.594094,1.222868,1.729396
10,P-0050430,2,1.662870,1.662870,0.864698,2.461042,1.128785,1.596344
7,P-0035844,3,1.928927,2.461042,0.864698,2.461042,0.921649,1.596344
5,P-0028971,2,1.297047,1.297047,0.864698,1.729396,0.611434,0.864698
3,P-0019531,2,0.432349,0.432349,0.000000,0.864698,0.611434,0.864698
14,P-0063459,2,0.410174,0.410174,0.000000,0.820347,0.580073,0.820347
18,P-0074869,2,5.332257,5.332257,4.922083,5.742431,0.580073,0.820347
13,P-0060318,3,1.367245,1.640694,0.820347,1.640694,0.473628,0.820347


# Save cleaned versions

In [19]:
# Create a processed data directory
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [20]:
# Save the sample-level data 
sample_df.to_csv(
    PROCESSED_DIR / "sample_level_cleaned.tsv",
    sep="\t",
    index=False,
)

In [21]:
# Save the survival ready data 
sample_survival_df.to_csv(
    PROCESSED_DIR / "sample_level_survival_ready.tsv",
    sep="\t",
    index=False,
)

In [22]:
# Confirm: 
list(PROCESSED_DIR.iterdir())

[PosixPath('/users/PAS3421/emmafischels/Capstone/data/processed/sample_level_cleaned.tsv'),
 PosixPath('/users/PAS3421/emmafischels/Capstone/data/processed/sample_level_survival_ready.tsv')]